In [1]:
%pip install uv

Note: you may need to restart the kernel to use updated packages.


In [22]:
!uv pip install --upgrade transformers datasets peft accelerate bitsandbytes qwen-vl-utils trackio
!uv pip install unsloth --torch-backend=auto
!uv pip install --upgrade ipywidgets widgetsnbextension jupyterlab_widgets

Using Python 3.12.14 environment at: /opt/homebrew/anaconda3/envs/splitr
Resolved 72 packages in 1.00s                                        
Prepared 20 packages in 7.91s                                                av                   ------------------------------ 17.34 MiB/17.37 MiB         av                   ------------------------------ 2.95 MiB/17.37 MiB          av                   ------------------------------ 1.26 MiB/17.37 MiB          av                   ------------------------------ 829.63 KiB/17.37 MiB        av                   ------------------------------ 317.63 KiB/17.37 MiB        av                   ------------------------------ 205.63 KiB/17.37 MiB        av                   ------------------------------ 125.63 KiB/17.37 MiB        av                   ------------------------------ 93.63 KiB/17.37 MiB         av                   ------------------------------ 48.00 KiB/17.37 MiB         Aav                   ------------------------------     0 B

In [37]:
from datasets import load_dataset

dataset = load_dataset("naver-clova-ix/cord-v2")
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [14]:
print(len(train_dataset))
print(len(eval_dataset))
print(len(test_dataset))

800
100
100


In [38]:
print(train_dataset[10])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=432x648 at 0x17D609700>, 'ground_truth': '{"gt_parse": {"menu": {"nm": "Viet Milk Coffee", "cnt": "1", "price": "25.000", "sub": [{"nm": "+Hot"}, {"nm": "+M"}]}, "sub_total": {"subtotal_price": "25.000"}, "total": {"total_price": "25.000", "cashprice": "30.000", "changeprice": "5.000"}}, "meta": {"version": "2.0.0", "split": "train", "image_id": 10, "image_size": {"width": 432, "height": 648}}, "valid_line": [{"words": [{"quad": {"x2": 170, "y3": 231, "x3": 171, "y4": 232, "x1": 162, "y1": 218, "x4": 162, "y2": 219}, "is_key": 0, "row_id": 1827478, "text": "1"}], "category": "menu.cnt", "group_id": 3, "sub_group_id": 0}, {"words": [{"quad": {"x2": 192, "y3": 232, "x3": 192, "y4": 232, "x1": 171, "y1": 218, "x4": 172, "y2": 218}, "is_key": 0, "row_id": 1827478, "text": "Viet"}, {"quad": {"x2": 218, "y3": 232, "x3": 217, "y4": 232, "x1": 192, "y1": 220, "x4": 192, "y2": 220}, "is_key": 0, "row_id": 1827478, "text": "Milk"}, {

In [39]:
system_prompt = "Extract all line items, quantities, and prices from this receipt as a JSON object"

def format_data(sample):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": sample["image"]},
                    {"type": "text", "text": system_prompt}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["ground_truth"]}
                ]
            }
        ]
    }

In [40]:
train_dataset = [format_data(sample) for sample in train_dataset]
eval_dataset = [format_data(sample) for sample in eval_dataset]
test_dataset = [format_data(sample) for sample in test_dataset]

In [18]:
train_dataset[200]

[{'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=864x1296>},
   {'type': 'text',
    'text': 'Extract all line items, quantities, and prices from this receipt as a JSON object'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': '{"gt_parse": {"menu": [{"nm": "NASI PUTIH", "unitprice": "6,000", "cnt": "1", "price": "6.000"}, {"nm": "BASO KUAH", "unitprice": "43,636", "cnt": "1", "price": "43,636"}], "sub_total": {"subtotal_price": "49,636", "tax_price": "4,964"}, "total": {"total_price": "54,600", "cashprice": "54.600", "changeprice": "0"}}, "meta": {"version": "2.0.0", "split": "train", "image_id": 200, "image_size": {"width": 864, "height": 1296}}, "valid_line": [{"words": [{"quad": {"x2": 174, "y3": 862, "x3": 174, "y4": 862, "x1": 108, "y1": 830, "x4": 108, "y2": 830}, "is_key": 0, "row_id": 2101887, "text": "NASI"}, {"quad": {"x2": 264, "y3": 862, "x3": 264, "y4": 862, "x1": 182, "y1": 832, "x4": 

In [19]:
import torch
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor

model_id = "Qwen/Qwen2-VL-7B-Instruct"

In [21]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = Qwen2VLProcessor.from_pretrained(model_id)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


...

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [50]:
from qwen_vl_utils import process_vision_info

def generate_text_from_sample(model, processor, sample, device, max_new_tokens=1024):
    # Extract only the user prompt turn (excluding target assistant response)
    user_messages = sample["messages"][:1]

    # Prepare text prompt with generation token
    text_input = processor.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Process vision inputs directly from the message structure
    image_inputs, video_inputs = process_vision_info(user_messages)

    # Move tensors to target device
    model_inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(device)

    # Generate response
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Strip prompt token IDs from output
    trimmed_generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decode generated tokens
    output_text = processor.batch_decode(
        trimmed_generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    return output_text[0]

In [51]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Device Count:", torch.cuda.device_count())

device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

CUDA Available: False
Device Count: 0
mps


In [52]:
output = generate_text_from_sample(model, processor, train_dataset[0], device=device)
output

KeyboardInterrupt: 